# ReefScan-Edge — GPU benchmark runner (Rungs 1–4)

Runs the inference-optimization harness on a Colab GPU. Self-contained: clones the public
repo and pulls the trained DINOv2-B checkpoint + the fixed 1,565-image test set from HF
(both public, no token needed).

**Before you start:** `Runtime -> Change runtime type -> L4 GPU` (or T4), then `Run all`.

Each rung registers a `(runtime, precision, device, batch)` variant into the harness, which
times with warmup + `cuda.synchronize()`, computes macro-F1 on the same test set, and appends
to `edge/results.csv` + `edge/RESULTS.md`. Re-running is **idempotent** — a rerun replaces a
variant's rows, it does not duplicate them, so `Run all` as many times as you like.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi -L

## 2. Clone repo + pinned GPU deps
torch/vision from the cu121 index (matches the pinned 2.4.1; Inductor/Triton ship inside it).
`numpy==1.26.4` is pinned <2 (onnxconverter-common's fp16 pass needs it, and it avoids numpy-2
ABI breaks across the torch/onnxruntime wheels). `transformers==4.44.2` is required — newer
versions route Dinov2 through SDPA, which torch 2.4.1's ONNX exporter mis-handles.

In [ ]:
!git clone https://github.com/HrishiKabra/reefscan.git
%cd reefscan
!pip install -q torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy==1.26.4 transformers==4.44.2 safetensors==0.4.5 huggingface_hub==0.25.2 \
                pyarrow==17.0.0 scikit-learn==1.5.2 pillow==10.4.0 matplotlib==3.9.2 \
                onnx==1.16.2 onnxruntime-gpu==1.19.2 onnxconverter-common==1.14.0

> If the next cell throws a NumPy/ABI error, do **Runtime -> Restart session**, then re-run
> from the `%cd reefscan` line below (Colab ships a preinstalled NumPy; a restart clears it).

In [ ]:
%cd /content/reefscan
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))

## 3. Rung 1 — PyTorch fp32 baseline (the control)

In [ ]:
!PYTHONPATH=. python -m edge.run_baseline

## 4. Rung 2 — torch.compile (Inductor, max-autotune)
First run compiles per input shape (a few minutes); that one-time cost is reported separately
and is NOT in the benchmarked latency. If `max-autotune` errors, set
`%env REEFSCAN_COMPILE_MODE=reduce-overhead` and re-run this cell.

In [ ]:
!PYTHONPATH=. python -m edge.run_rung2

## 5. Rung 3 — ONNX export + ONNX Runtime (CUDA EP)
Point ORT at the CUDA/cuDNN libs torch already installed (matched versions); the `!` cell runs
as a fresh subprocess and inherits this `LD_LIBRARY_PATH`. The script hard-fails if the CUDA EP
didn't load, so a `cuda` row can never be mislabeled CPU latency.

In [ ]:
import os, glob, torch
libs = sorted(glob.glob('/usr/local/lib/python3.*/dist-packages/nvidia/*/lib'))
libs.append(os.path.join(os.path.dirname(torch.__file__), 'lib'))
os.environ['LD_LIBRARY_PATH'] = ':'.join(libs + [os.environ.get('LD_LIBRARY_PATH', '')])
print('LD_LIBRARY_PATH set to include:')
print('\n'.join(libs))

In [ ]:
!PYTHONPATH=. python -m edge.run_rung3

## 6. Rung 3b — precision: fp16 + int8 PTQ + TF32 control
Three variants: **pytorch-tf32** (isolates how much of the ORT win was TF32 vs fusion),
**onnxruntime-fp16** (fp16 compute, fp32 IO — full tensor-core throughput; lossless), and
**onnxruntime-int8** (static PTQ, representative-subsample calibration).

**Expected:** the int8 row's macro-F1 **collapses to ~0.43–0.48** — a known, documented negative
(static int8 PTQ squashes ViT activation outliers). It's recorded honestly as a measured row;
viable int8 is QAT / TensorRT entropy-calibrated int8 (Rung 4). Don't be alarmed by the low int8
number. int8 latency is on CPU (ORT's CUDA EP has no int8). Runs after Rung 3 (reuses its ONNX).

In [ ]:
!PYTHONPATH=. python -m edge.run_rung3b

## 7. Rung 4 — TensorRT (fp16 + int8 entropy calibration)
Builds TensorRT engines from the Rung-3 ONNX. **tensorrt-fp16** (TRT fuses the whole transformer +
autotunes kernels for this GPU — should match/beat ORT fp16) and **tensorrt-int8** (calibrated with
`IInt8EntropyCalibrator2` on the representative subsample). This is the payoff to the Rung-3b int8
negative: entropy calibration + int8 tensor-core kernels are the viable int8 path.

Engine builds take a few minutes each (int8 longer — it calibrates + autotunes). The cell adds
`tensorrt_libs` to `LD_LIBRARY_PATH` (inherited by the run subprocess).

In [ ]:
!pip install -q tensorrt==10.5.0
import os, glob
trt_libs = glob.glob('/usr/local/lib/python3.*/dist-packages/tensorrt_libs')
os.environ['LD_LIBRARY_PATH'] = ':'.join(trt_libs + [os.environ.get('LD_LIBRARY_PATH', '')])
print('added to LD_LIBRARY_PATH:', trt_libs)

In [ ]:
!PYTHONPATH=. python -m edge.run_rung4

## 8. Pareto frontier — accuracy vs latency (the centerpiece figure)

In [ ]:
!PYTHONPATH=. python -m edge.plot_pareto
from IPython.display import Image
Image("edge/docs/pareto.png")

## 9. The results table (CPU local + GPU rungs)

In [ ]:
print(open("edge/RESULTS.md").read())

## 10. Send the numbers back
Paste this cell's output back into the chat — it gets committed. (Download cells below for the
CSV + the Pareto PNG.)

In [ ]:
print(open("edge/results.csv").read())

In [ ]:
# optional: download the CSV + the Pareto figure
from google.colab import files
files.download("edge/results.csv")
files.download("edge/docs/pareto.png")